# Harvest Now, Decrypt Later (Shor + PQC)

*Secure the Grid, step 2 · UTC Problem 4*

This problem is mostly analysis. The code here is a small proof of concept: factor 15 with Shor's algorithm. Parts 3 and 4 (exposure map and migration plan) are written — do them in your write-up.

> **Before you run anything, pick the kernel.** When this notebook opens, a **Select Kernel** box appears with *Python 3 (ipykernel)* already selected. That's the wrong one: change it to **qBraid-SDK (v0.12.2)**, then click **Select**. Picked the wrong one, or closed the box? Click the kernel name at the top right of the notebook and choose **qBraid-SDK (v0.12.2)**. The first cell checks, and tells you if it's wrong.

In [ ]:
# Use the qBraid-SDK (v0.12.2) kernel. This stops with a clear message if another one is picked.
import importlib.util
missing = [m for m in ("qbraid", "matplotlib") if importlib.util.find_spec(m) is None]
if missing:
    raise ImportError("Wrong kernel: this one has no " + " or ".join(missing) + ". Click the kernel name "
                      "at the top right of the notebook, pick qBraid-SDK (v0.12.2), then run this cell again.")

# qBraid Lab: install Qiskit Aer if this environment doesn't have it yet (one time, ~20 s)
try:
    import qiskit_aer  # noqa: F401
except ImportError:
    %pip install -q qiskit-aer

import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
sim = AerSimulator()

print("Setup OK: the qBraid kernel and Qiskit Aer are ready. You're good to go.")

### Part 1 — Shor demo (factor 15)

The circuit below performs quantum period-finding for N = 15 with base a = 7 (a standard textbook case). Run it, read the measured period, and use it to recover the factors. The classical post-processing is left as a short `TODO`.

> **Before this goes on a slide:** look at `c_amod15_a7` in the next cell. Would it work for N = 21, or for a 2048-bit number? What does that say about what this demo proves, and what it doesn't?


In [ ]:
def c_amod15_a7(power):
    """Controlled multiplication by 7^power mod 15 (precompiled for the demo)."""
    U = QuantumCircuit(4)
    for _ in range(power):
        U.swap(0, 1); U.swap(1, 2); U.swap(2, 3)
        for q in range(4):
            U.x(q)
    U = U.to_gate(); U.name = f"7^{power} mod 15"
    return U.control()

def qft_dagger(n):
    qc = QuantumCircuit(n)
    for q in range(n // 2):
        qc.swap(q, n - q - 1)
    for j in range(n):
        for m in range(j):
            qc.cp(-np.pi / 2 ** (j - m), m, j)
        qc.h(j)
    qc.name = "QFT†"
    return qc


In [ ]:
n_count = 8                      # counting qubits
qc = QuantumCircuit(n_count + 4, n_count)
qc.h(range(n_count))
qc.x(n_count)                    # set work register to |1>
for i in range(n_count):
    qc.append(c_amod15_a7(2 ** i), [i] + list(range(n_count, n_count + 4)))
qc.append(qft_dagger(n_count), range(n_count))
qc.measure(range(n_count), range(n_count))

counts = sim.run(transpile(qc, sim), shots=1024).result().get_counts()
measured = max(counts, key=counts.get)
phase = int(measured, 2) / (2 ** n_count)
print("Measured phase:", phase)


In [ ]:
from fractions import Fraction
from math import gcd

# Each run lands on one of four phases (0, 1/4, 1/2, 3/4) about equally often.
# Phases 0 and 1/2 give a wrong period, so try the results from most to least common
# and keep the first r that is even and passes the check 7^r mod 15 = 1.
ranked = sorted(counts, key=counts.get, reverse=True)
for measured in ranked:
    phase = int(measured, 2) / (2 ** n_count)
    r = Fraction(phase).limit_denominator(15).denominator
    print(f"measured {measured} -> phase {phase} -> candidate period r = {r}, 7^r mod 15 = {pow(7, r, 15)}")
    if r % 2 == 0 and pow(7, r, 15) == 1:
        break

# TODO: use the period r to compute the factors of 15.
# Hint: the factors are gcd(7**(r//2) - 1, 15) and gcd(7**(r//2) + 1, 15). Print them.
# When your lines are in, delete the raise line below.
raise NotImplementedError("Recover the two factors of 15 from r")

### Part 2 — how far is real hardware?

List the real devices on qBraid and record their qubit counts. Then estimate how many qubits breaking RSA-2048 would need, and state the gap.


In [ ]:
# List qBraid's gate-based quantum computers and their qubit counts
from qbraid.runtime import QbraidProvider

provider = QbraidProvider()
analog = ("quera:qpu:aquila", "pasqal:qpu:fresnel")      # analog machines: they can't run Shor's circuits
for d in provider.get_devices():
    if ":qpu:" in d.id and not d.id.endswith(analog):
        print(f"{d.id:42s} {d.num_qubits} qubits")
# The same machine can appear under several routes (direct, aws, azure, openquantum): count it once.

# Then, in your write-up:
#  - largest real device qubit count you found: ______
#  - qubits needed to break RSA-2048: ______ logical, ______ physical
#    Cite the source, year and assumptions: the estimate has fallen a lot. Start with Gidney & Ekerå
#    (2019) and Gidney (2025, arXiv:2505.15917), then look for anything newer.
#  - the gap: ______ orders of magnitude in qubit count. What does a qubit count leave out?

## What to have ready
Written answers go in `WRITEUP.md` in this folder.
- **Part 1:** the factors of 15 recovered from the period.
- **Part 2:** the device-list qubit counts and your gap estimate.
- **Part 3:** the exposure map (written).
- **Part 4:** the migration plan with priority ranking (written).
- **Part 5:** next, open `2b_grover_symmetric_keys.ipynb` (Grover's search and symmetric keys).
- **For your deck:** the gap between today's machines and RSA-2048, and how fast it's shrinking, sets the urgency of your migration plan.
